# TOPIC 10 – ĐO LƯỜNG RỦI RO DANH MỤC VN30 BẰNG DCC-GARCH
## Thành viên 4 – Quỳnh: VaR, ES, Backtest (trong mẫu + ngoài mẫu)
Chạy **sau** TV3. Chạy bằng **Restart Kernel → Run All**. Thứ tự chung của cả nhóm: TV1 → TV2 → TV3 → TV4 → TV5.

| | |
|---|---|
| Đọc từ | `TV1_Data_EDA/log_returns.csv`, `TV2_GARCH/garch_selected_models.csv`, `TV3_DCC/dcc_matrices.npz` |
| Ghi ra | `TV4_VaR_Backtest/` (`TV4_var_backtest.xlsx`, `TV4_out_of_sample.xlsx`, `oos_*.csv`, `tv4_settings.json`, hình) |
| Cấu hình | ô 0.2 (đường dẫn, hằng số) – **giống hệt ở cả 5 notebook**, chỉ cần sửa `PROJECT_PATH` / `OUTPUT_PATH` cho đúng máy |

Thư viện: numpy, pandas, scipy, matplotlib, openpyxl (statsmodels tuỳ chọn). Xem `README.md`.

In [ ]:
# ===================== 0.1 THƯ VIỆN =====================
import os                     # đọc biến môi trường
import re                     # biểu thức chính quy: tách dòng cổ tức / chia tách trong file Yahoo
import warnings               # tắt cảnh báo không cần thiết
import datetime as dt         # kiểu ngày tháng
from itertools import combinations   # tạo các cặp cổ phiếu
from pathlib import Path      # thao tác đường dẫn an toàn trên Windows

import numpy as np            # tính toán ma trận
import pandas as pd           # bảng dữ liệu
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats                     # phân phối Normal / t / chi-bình phương, JB
from scipy.optimize import minimize         # tối đa hoá hàm hợp lý (MLE)
from scipy.signal import lfilter            # lọc tuyến tính nhanh cho phương trình GARCH
from scipy.special import gammaln           # log Gamma cho mật độ Student-t

try:                                        # statsmodels: chỉ dùng cho ADF / KPSS (nếu có)
    from statsmodels.tsa.stattools import adfuller, kpss
    from statsmodels.tools.sm_exceptions import InterpolationWarning
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:                                        # display() có sẵn trong Jupyter; chạy file .py thì thay bằng print
    display
except NameError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
print("statsmodels có sẵn:", HAS_SM)

In [ ]:
# ===================== 0.2 CẤU HÌNH DÙNG CHUNG CHO CẢ 5 THÀNH VIÊN =====================
# >>> ĐƯỜNG DẪN – sửa 2 dòng này nếu đổi máy / đổi thư mục <<<
PROJECT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo"          # thư mục dự án (chứa input/, ouput/, code/)
OUTPUT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo\ouput"    # nơi lưu toàn bộ kết quả


def find_base_dir() -> Path:
    """Tìm thư mục dự án: biến môi trường PROJECT_DIR -> PROJECT_PATH -> thư mục cha của code/."""
    env = os.environ.get("PROJECT_DIR")
    if env:
        return Path(env)
    if Path(PROJECT_PATH).exists():
        return Path(PROJECT_PATH)
    cwd = Path.cwd()
    return cwd.parent if cwd.name.lower() == "code" else cwd


def pick_dir(base: Path, names) -> Path:
    """Chọn thư mục con đã tồn tại (chấp nhận Input/input)."""
    return next((base / n for n in names if (base / n).exists()), base / names[0])


BASE_DIR = find_base_dir()
INPUT_DIR = pick_dir(BASE_DIR, ["input", "Input"])
if not list(INPUT_DIR.glob("*.xlsx")):                  # dự phòng: thư mục Input cũ của Project midterm
    old_in = Path(r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\Project midterm\Input")
    if list(old_in.glob("*.xlsx")):
        INPUT_DIR = old_in
# Output: biến môi trường OUTPUT_DIR -> OUTPUT_PATH (nếu ổ D: tồn tại) -> BASE_DIR/ouput
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR") or
                  (OUTPUT_PATH if Path(OUTPUT_PATH).anchor and Path(Path(OUTPUT_PATH).anchor).exists()
                   else BASE_DIR / "ouput"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_OUT = OUTPUT_DIR                                    # mỗi thành viên một thư mục con trong ouput/
OUT = {k: ROOT_OUT / v for k, v in {"TV1": "TV1_Data_EDA", "TV2": "TV2_GARCH", "TV3": "TV3_DCC",
                                     "TV4": "TV4_VaR_Backtest", "TV5": "TV5_Euler_Stress"}.items()}
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)

# ---- Hằng số dùng chung ----
TICKERS_ORDER = ["FPT", "HPG", "MWG", "VCB", "VHM"]    # thứ tự hiển thị (mã khác trong Input vẫn được đọc)
ALPHA = 0.05                 # mức ý nghĩa của mọi kiểm định
LB_LAGS = [5, 10, 20]        # số trễ Ljung-Box
TRADING_DAYS = 252           # số phiên / năm để năm hoá
SCALE = 100.0                # ước lượng GARCH trên lợi suất % (r × 100) cho ổn định số học
SELECT_BY = "BIC"            # tiêu chí chọn mô hình GARCH: "AIC" hoặc "BIC"
PAIRS = [("VCB", "VHM"), ("HPG", "FPT")]               # cặp tương quan động theo đề
VAR_LEVELS = [0.95, 0.99]    # mức tin cậy VaR
HS_WINDOW = 250              # cửa sổ Historical Simulation (≈ 1 năm)
BASEL_WINDOW = 250           # cửa sổ Basel Traffic Light (250 phiên gần nhất)
VIOL_MODEL = "Normal"        # >>> MÔ HÌNH ĐÁNH DẤU VI PHẠM TRÊN MỌI HÌNH & DIỄN GIẢI TV4: "Normal" hoặc "t" <<<
VIOL_OOS = {"Normal": "DCC-N", "t": "DCC-t"}[VIOL_MODEL]   # tên tương ứng trong phần ngoài mẫu
EULER_CL = 0.99              # mức tin cậy dùng cho phân bổ rủi ro Euler
STRESS = {                   # giai đoạn khủng hoảng theo đề (TV5)
    "Covid-19 (02–04/2020)": ("2020-02-01", "2020-04-30"),
    "Trái phiếu & lãi suất (04–11/2022)": ("2022-04-01", "2022-11-30"),
}


STRESS_COLORS = ["tab:red", "tab:orange"]   # mỗi giai đoạn một màu để phân biệt Covid-19 với Trái phiếu & lãi suất


def shade_stress(ax, alpha=0.12, label=True, xlim=None):
    """Tô màu từng giai đoạn khủng hoảng (mỗi giai đoạn một màu, chú thích theo tên giai đoạn).
    xlim=(đầu, cuối): bỏ qua giai đoạn nằm hoàn toàn ngoài khoảng này (tránh kéo giãn trục thời gian)."""
    for (name, (s, e)), c in zip(STRESS.items(), STRESS_COLORS):
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        if xlim is not None and (e < xlim[0] or s > xlim[1]):
            continue
        ax.axvspan(s, e, color=c, alpha=alpha, label=(name if label else None))


def grid_axes(n, ncol=2, h=3.2, w=13, **kw):
    """Tạo lưới ô vẽ cho n mã, ẩn các ô thừa."""
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w, h * nrow), squeeze=False, **kw)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


print("BASE_DIR  :", BASE_DIR)
print("INPUT_DIR :", INPUT_DIR, "| tồn tại:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


def period_masks(index):
    """Mặt nạ True/False cho từng giai đoạn: các giai đoạn STRESS + 'Bình thường' (ngoài mọi giai đoạn stress)."""
    m = {name: (index >= s) & (index <= e) for name, (s, e) in STRESS.items()}
    m = {"Bình thường": ~np.any(np.column_stack(list(m.values())), axis=1), **m}
    return m


## Nạp kết quả của các thành viên trước

In [ ]:
# ===================== NẠP KẾT QUẢ TV1, TV2, TV3 =====================
def need(path, prev):
    """Báo lỗi rõ ràng nếu chưa chạy notebook của thành viên trước."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Chưa có {path} – hãy chạy notebook {prev} trước (xem README.md).")
    return Path(path)


log_ret = pd.read_csv(need(OUT["TV1"] / "log_returns.csv", "TV1"), index_col=0, parse_dates=True, float_precision="round_trip")
log_ret.index.name = "Date"
tickers = list(log_ret.columns); N = len(tickers); T = len(log_ret)
print(f"[TV1] log_ret: {T} phiên × {N} mã ({', '.join(tickers)}) | {log_ret.index.min().date()} -> {log_ret.index.max().date()}")
sel_h = pd.read_csv(need(OUT["TV2"] / "garch_selected_models.csv", "TV2"), index_col=0, float_precision="round_trip").loc[tickers]
best_name = sel_h["Mô hình"]
SPEC = {tk: dict(gjr=bool(sel_h.loc[tk, "gjr"]), dist=str(sel_h.loc[tk, "dist"])) for tk in tickers}   # mô hình biên cho phần ngoài mẫu
z_dcc = np.load(need(OUT["TV3"] / "dcc_matrices.npz", "TV3"), allow_pickle=False)
assert list(pd.to_datetime(z_dcc["dates"])) == list(log_ret.index) and [str(t) for t in z_dcc["tickers"]] == tickers, \
    "Output TV1 và TV3 không khớp ngày/mã – chạy lại từ TV1."
R_t, H_t, R_fcst, H_fcst = z_dcc["R"], z_dcc["H"], z_dcc["R_fcst"], z_dcc["H_fcst"]
a_hat, b_hat = float(z_dcc["a"]), float(z_dcc["b"])
ok_stat = (a_hat >= 0) and (b_hat >= 0) and (a_hat + b_hat < 1)
S = np.sqrt(np.einsum("tii->ti", H_t))      # σ_t từng mã = sqrt(đường chéo H_t)
idx_tk = {tk: i for i, tk in enumerate(tickers)}
masks = period_masks(log_ret.index)
iu_ = np.triu_indices(N, 1)
avg_corr = pd.Series(R_t[:, iu_[0], iu_[1]].mean(axis=1), index=log_ret.index)      # tương quan DCC trung bình các cặp
print(f"[TV3] H_t {H_t.shape} | a = {a_hat:.4f}, b = {b_hat:.4f}")

---
# THÀNH VIÊN 4: ĐO LƯỜNG RỦI RO DANH MỤC & BACKTESTING (VaR, ES & BASEL TESTS)
## 4.1 Biến động có điều kiện của danh mục
*(Mục 4.5 bổ sung dự báo VaR/ES **ngoài mẫu** với ước lượng lại theo cửa sổ trượt.)*
**Tầng 1:** Tỷ trọng bằng nhau $w = [0{,}2;\,0{,}2;\,0{,}2;\,0{,}2;\,0{,}2]'$; $\sigma_{p,t} = \sqrt{w'H_tw}$ với $H_t$ của DCC (TV3).
Lợi suất danh mục $r_{p,t} = w'r_t$. Lợi ích đa dạng hoá $= 1 - \sigma_{p,t}/\sum_i w_i\sigma_{i,t}$ (0% khi mọi tương quan = 1).

**Tầng 2:** Input `H_t`, `log_ret`. Output `sigma_p` (Series), `r_p` (Series). Có thể đổi `WEIGHTS` (ví dụ theo vốn hoá) – tổng phải bằng 1.

**Tầng 3:** σ_p thấp hơn trung bình σ từng mã nhờ đa dạng hoá; khoảng cách này thu hẹp trong khủng hoảng khi tương quan tăng.

In [ ]:
# ===================== 4.1 BIẾN ĐỘNG DANH MỤC =====================
WEIGHTS = pd.Series(1 / N, index=tickers)             # Equal weight; muốn theo vốn hoá thì gán Series khác, tổng = 1
assert np.isclose(WEIGHTS.sum(), 1), "Tổng tỷ trọng phải bằng 1"
w = WEIGHTS.values
r_p = pd.Series(log_ret.values @ w, index=log_ret.index, name="r_p")
sigma_p = pd.Series(np.sqrt(np.einsum("i,tij,j->t", w, H_t, w)), index=log_ret.index, name="sigma_p")
undiv = pd.Series(S @ w, index=log_ret.index)          # σ danh mục nếu mọi tương quan = 1
div_benefit = 1 - sigma_p / undiv
print("Tỷ trọng:", WEIGHTS.round(3).to_dict())
print(f"σ_p trung bình năm hoá: {sigma_p.mean() * np.sqrt(TRADING_DAYS):.2%} | lợi ích đa dạng hoá TB: {div_benefit.mean():.1%}")

fig, ax = plt.subplots(figsize=(13, 4))
shade_stress(ax)
ax.plot(r_p.index, r_p.abs() * np.sqrt(TRADING_DAYS), color="lightgrey", lw=0.5, label="|r_p| năm hoá")
ax.plot(sigma_p.index, sigma_p * np.sqrt(TRADING_DAYS), color="tab:red", lw=1.1, label="σ_p DCC năm hoá")
ax.plot(undiv.index, undiv * np.sqrt(TRADING_DAYS), color="tab:blue", lw=0.9, ls="--", label="Σ w_i σ_i (không đa dạng hoá)")
ax.set_title("Biến động có điều kiện của danh mục"); ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_ylim(0, None)
plt.tight_layout(); plt.savefig(OUT["TV4"] / "4_portfolio_volatility.png", dpi=150); plt.show()

## 4.2 VaR & ES 1 ngày: Normal, Student-t, Historical Simulation
**Tầng 1** (quy ước VaR, ES là **tổn thất dương**, lợi suất danh mục trung bình = 0; α = 5% hoặc 1%):
- Normal: $VaR_{\alpha,t} = z_{1-\alpha}\,\sigma_{p,t}$; $ES_{\alpha,t} = \sigma_{p,t}\,\dfrac{\phi(z_{1-\alpha})}{\alpha}$.
- Student-t chuẩn hoá (phương sai 1, ν ước lượng bằng MLE trên $z_{p,t} = r_{p,t}/\sigma_{p,t}$), $c = \sqrt{(\nu-2)/\nu}$, $q = t_\nu^{-1}(1-\alpha)$:
  $VaR = c\,q\,\sigma_{p,t}$; $ES = c\,\sigma_{p,t}\,\dfrac{f_\nu(q)}{\alpha}\cdot\dfrac{\nu + q^2}{\nu - 1}$.
- Historical Simulation: VaR = phân vị α của 250 lợi suất danh mục **trước** ngày t (đổi dấu); ES = trung bình các khoản lỗ vượt VaR trong cửa sổ.

**Tầng 2:** Output `var_es` (DataFrame theo ngày: σ_p, r_p, VaR_*, ES_*). Cửa sổ backtest bắt đầu sau 250 phiên đầu (để HS có đủ dữ liệu)
nên cả 3 phương pháp được so sánh trên cùng các ngày. **Lưu ý:** tham số GARCH/DCC ước lượng trên toàn mẫu (in-sample) ⇒ backtest này
kiểm tra độ phù hợp của mô hình; dự báo ngoài mẫu thật (ước lượng lại theo cửa sổ trượt) nằm ở mục 4.5.

**Tầng 3:** ES luôn > VaR cùng mức tin cậy. Ở 99%, VaR-t > VaR-Normal (đuôi dày); ở 95% có thể ngược lại vì t chuẩn hoá "nhọn" hơn ở giữa.

In [ ]:
# ===================== 4.2a ƯỚC LƯỢNG ν CHO STUDENT-t =====================
z_p = (r_p / sigma_p).values


def std_t_nll(nu, x):
    """−log-likelihood của Student-t chuẩn hoá (phương sai 1)."""
    c = np.sqrt((nu - 2) / nu)
    return -np.sum(stats.t.logpdf(x / c, nu) - np.log(c))


nu_p = minimize(lambda v: std_t_nll(v[0], z_p), [6.0], bounds=[(2.05, 100)], method="L-BFGS-B").x[0]
print(f"Phần dư danh mục z_p: std = {z_p.std():.3f}, kurtosis = {stats.kurtosis(z_p, fisher=False):.2f} -> ν ước lượng = {nu_p:.2f}")

In [ ]:
# ===================== 4.2b TÍNH VaR / ES =====================
def var_es_normal(sig, a):
    z = stats.norm.ppf(1 - a)
    return z * sig, sig * stats.norm.pdf(z) / a


def var_es_t(sig, a, nu):
    c = np.sqrt((nu - 2) / nu)
    q = stats.t.ppf(1 - a, nu)
    return c * q * sig, c * sig * stats.t.pdf(q, nu) / a * (nu + q ** 2) / (nu - 1)


def var_es_hs(r, a, window):
    """HS: dùng window lợi suất TRƯỚC ngày t (không nhìn trước)."""
    var, es = np.full(len(r), np.nan), np.full(len(r), np.nan)
    x = r.values
    for t in range(window, len(x)):
        hist = x[t - window:t]
        q = np.quantile(hist, a)
        var[t], es[t] = -q, -hist[hist <= q].mean()
    return var, es


var_es = pd.DataFrame({"r_p": r_p, "loss": -r_p, "sigma_p": sigma_p})
for cl in VAR_LEVELS:
    a = 1 - cl
    tag = f"{int(cl * 100)}"
    var_es[f"VaR_Normal_{tag}"], var_es[f"ES_Normal_{tag}"] = var_es_normal(sigma_p.values, a)
    var_es[f"VaR_t_{tag}"], var_es[f"ES_t_{tag}"] = var_es_t(sigma_p.values, a, nu_p)
    var_es[f"VaR_HS_{tag}"], var_es[f"ES_HS_{tag}"] = var_es_hs(r_p, a, HS_WINDOW)
bt = var_es.iloc[HS_WINDOW:].copy()                    # cửa sổ backtest chung cho 3 phương pháp
print(f"Cửa sổ backtest: {bt.index.min().date()} -> {bt.index.max().date()} ({len(bt)} phiên)")
display((bt.filter(regex="^(VaR|ES)").mean() * 100).round(3).to_frame("Trung bình (%)"))

METHODS = ["Normal", "t", "HS"]
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, cl in zip(axes, VAR_LEVELS):
    tag = f"{int(cl * 100)}"
    loss = -bt["r_p"]
    ax.plot(bt.index, loss * 100, color="lightgrey", lw=0.6, label="Tổn thất thực tế L_t")
    for m, c in zip(METHODS, ["tab:red", "tab:purple", "tab:green"]):
        ax.plot(bt.index, bt[f"VaR_{m}_{tag}"] * 100, lw=0.9, color=c, label=f"VaR {m}")
    br = loss > bt[f"VaR_{VIOL_MODEL}_{tag}"]                        # mô hình đánh dấu vi phạm: VIOL_MODEL (ô 0.2)
    ax.scatter(bt.index[br], loss[br] * 100, color="black", s=10, zorder=3, label=f"Vi phạm (VaR {VIOL_MODEL}) – {int(br.sum())}")
    shade_stress(ax, alpha=0.08)
    ax.set_title(f"VaR {tag}% 1 ngày (DCC-GARCH) và tổn thất thực tế"); ax.set_ylabel("%"); ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout(); plt.savefig(OUT["TV4"] / "4_var_backtest.png", dpi=150, bbox_inches="tight"); plt.show()


## 4.3 Kiểm định ngược (Backtesting)
**Tầng 1:**
- Vi phạm: $I_t = 1$ nếu tổn thất $-r_{p,t} > VaR_t$. Kỳ vọng $n\alpha$ lần.
- **Kupiec POF** ($H_0$: tỷ lệ vi phạm $p = \alpha$): $LR_{uc} = -2\ln\dfrac{(1-\alpha)^{n-x}\alpha^x}{(1-\hat p)^{n-x}\hat p^x} \sim \chi^2(1)$.
- **Christoffersen** ($H_0$: vi phạm độc lập): $LR_{ind} = -2\ln\dfrac{(1-\pi)^{n_{00}+n_{10}}\pi^{n_{01}+n_{11}}}{(1-\pi_{01})^{n_{00}}\pi_{01}^{n_{01}}(1-\pi_{11})^{n_{10}}\pi_{11}^{n_{11}}} \sim \chi^2(1)$;
  kết hợp $LR_{cc} = LR_{uc} + LR_{ind} \sim \chi^2(2)$.
- **Basel Traffic Light** (VaR 99%, 250 phiên gần nhất): Xanh 0–4, Vàng 5–9, Đỏ ≥ 10 vi phạm; hệ số nhân vốn k = 3 + plus factor.
  Với toàn bộ cửa sổ, vùng được xác định theo xác suất nhị thức tích luỹ (Xanh < 95%, Vàng < 99,99%, Đỏ còn lại) – tương ứng ngưỡng Basel.
- Kiểm tra ES: so sánh tổn thất thực tế trung bình trong các ngày vi phạm với ES dự báo trung bình những ngày đó.

**Tầng 3:** tỷ lệ vi phạm > α và Kupiec p < 0,05 ⇒ **đánh giá thấp** rủi ro; tỷ lệ < α và p < 0,05 ⇒ **thổi phồng** rủi ro (giữ vốn thừa).
Christoffersen p < 0,05 ⇒ vi phạm **thành cụm**: mô hình phản ứng chậm khi biến động tăng đột ngột.

In [ ]:
# ===================== 4.3 CÁC HÀM KIỂM ĐỊNH =====================
def _xlogy(x, y):
    return 0.0 if x == 0 else x * np.log(y)


def kupiec_pof(hits, a):
    n, x = len(hits), int(hits.sum())
    p_hat = x / n
    ll0 = _xlogy(n - x, 1 - a) + _xlogy(x, a)
    ll1 = _xlogy(n - x, 1 - p_hat) + _xlogy(x, p_hat)
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1)


def christoffersen_ind(hits):
    h = np.asarray(hits, int)
    prev, cur = h[:-1], h[1:]
    n00 = int(((prev == 0) & (cur == 0)).sum()); n01 = int(((prev == 0) & (cur == 1)).sum())
    n10 = int(((prev == 1) & (cur == 0)).sum()); n11 = int(((prev == 1) & (cur == 1)).sum())
    pi01 = n01 / max(n00 + n01, 1); pi11 = n11 / max(n10 + n11, 1)
    pi = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    ll0 = _xlogy(n00 + n10, 1 - pi) + _xlogy(n01 + n11, pi)
    ll1 = _xlogy(n00, 1 - pi01) + _xlogy(n01, pi01) + _xlogy(n10, 1 - pi11) + _xlogy(n11, pi11)
    lr = -2 * (ll0 - ll1)
    return lr, stats.chi2.sf(lr, 1), n11


def binomial_zone(x, n, a):
    cdf = stats.binom.cdf(x, n, a)                              # P(X ≤ x): trùng bảng Basel 0–4 / 5–9 / ≥10 khi n = 250, α = 1%
    return "Xanh" if cdf < 0.95 else ("Vàng" if cdf < 0.9999 else "Đỏ")


BASEL_PLUS = {5: 0.40, 6: 0.50, 7: 0.65, 8: 0.75, 9: 0.85}

rows = []
for cl in VAR_LEVELS:
    a = 1 - cl
    tag = f"{int(cl * 100)}"
    for m in METHODS:
        loss = -bt["r_p"]
        hits = (loss > bt[f"VaR_{m}_{tag}"]).astype(int).values
        n, x = len(hits), int(hits.sum())
        lr_uc, p_uc = kupiec_pof(hits, a)
        lr_ind, p_ind, n11 = christoffersen_ind(hits)
        lr_cc = lr_uc + lr_ind
        hit_days = hits.astype(bool)
        rows.append({"Mức tin cậy": f"{tag}%", "Phương pháp": m, "Số ngày": n, "Vi phạm": x,
                     "Kỳ vọng": round(n * a, 1), "Tỷ lệ vi phạm (%)": 100 * x / n,
                     "Kupiec LR": lr_uc, "Kupiec p": p_uc, "Christoffersen LR": lr_ind, "Christoffersen p": p_ind,
                     "Vi phạm liên tiếp (n11)": n11, "LR_cc": lr_cc, "LR_cc p": stats.chi2.sf(lr_cc, 2),
                     "Vùng (nhị thức)": binomial_zone(x, n, a),
                     "Lỗ TB ngày vi phạm (%)": 100 * loss[hit_days].mean() if x else np.nan,
                     "ES dự báo TB ngày vi phạm (%)": 100 * bt[f"ES_{m}_{tag}"][hit_days].mean() if x else np.nan})
bt_table = pd.DataFrame(rows)


def verdict(row, a):
    over = row["Tỷ lệ vi phạm (%)"] > 100 * a
    s = ("ĐÁNH GIÁ THẤP rủi ro" if over else "THỔI PHỒNG rủi ro") if row["Kupiec p"] < ALPHA else "Tỷ lệ vi phạm phù hợp"
    s += "; vi phạm THÀNH CỤM" if row["Christoffersen p"] < ALPHA else "; vi phạm độc lập"
    return s


bt_table["Kết luận"] = [verdict(r, 1 - float(r["Mức tin cậy"][:-1]) / 100) for _, r in bt_table.iterrows()]
display(bt_table.round(4))

# Basel Traffic Light chuẩn: VaR 99%, 250 phiên cuối
last = bt.iloc[-250:]
basel_rows = []
for m in METHODS:
    x = int((-last["r_p"] > last[f"VaR_{m}_99"]).sum())
    zone = "Xanh" if x <= 4 else ("Vàng" if x <= 9 else "Đỏ")
    plus = 0.0 if x <= 4 else BASEL_PLUS.get(x, 1.0)
    basel_rows.append({"Phương pháp": m, "Cửa sổ": f"{last.index.min().date()} -> {last.index.max().date()}",
                       "Vi phạm VaR99 (250 phiên)": x, "Vùng Basel": zone, "Plus factor": plus, "Hệ số nhân vốn k": 3 + plus})
basel_table = pd.DataFrame(basel_rows)
display(basel_table)

In [ ]:
# ===================== 4.3b VI PHẠM THEO NĂM / GIAI ĐOẠN =====================
hits_df = pd.DataFrame({f"{m}_{int(cl * 100)}": (-bt["r_p"] > bt[f"VaR_{m}_{int(cl * 100)}"]).astype(int)
                        for cl in VAR_LEVELS for m in METHODS}, index=bt.index)
by_year = hits_df.groupby(hits_df.index.year).sum()
by_year["Số phiên"] = hits_df.groupby(hits_df.index.year).size()
print("Số lần vi phạm theo năm:"); display(by_year)
bt_masks = period_masks(bt.index)
rate_period = pd.DataFrame({name: hits_df[m].mean() * 100 for name, m in bt_masks.items() if m.any()}).T
rate_period["Số phiên"] = [int(m.sum()) for name, m in bt_masks.items() if m.any()]
print("Tỷ lệ vi phạm theo giai đoạn (%, kỳ vọng 5% / 1%):"); display(rate_period.round(2))

with pd.ExcelWriter(OUT["TV4"] / "TV4_var_backtest.xlsx", engine="openpyxl") as xw:
    bt_table.to_excel(xw, sheet_name="Backtest", index=False)
    basel_table.to_excel(xw, sheet_name="Basel_250", index=False)
    (bt.filter(regex="^(VaR|ES)").mean() * 100).to_frame("Trung bình (%)").to_excel(xw, sheet_name="VaR_ES_TB")
    by_year.to_excel(xw, sheet_name="Vi_pham_theo_nam")
    rate_period.to_excel(xw, sheet_name="Vi_pham_theo_giai_doan")
    var_es.to_excel(xw, sheet_name="VaR_ES_hang_ngay")
print("✔ Đã lưu kết quả TV4 vào", OUT["TV4"])

## 4.4 Diễn giải cốt lõi – DCC-GARCH có đo lường đúng rủi ro danh mục?

In [ ]:
# ===================== 4.4 DIỄN GIẢI TỰ ĐỘNG =====================
for _, r in bt_table.iterrows():
    print(f"VaR {r['Mức tin cậy']} – {r['Phương pháp']:<6}: {r['Vi phạm']} vi phạm / kỳ vọng {r['Kỳ vọng']} "
          f"({r['Tỷ lệ vi phạm (%)']:.2f}%) | Kupiec p = {r['Kupiec p']:.3f}, Christoffersen p = {r['Christoffersen p']:.3f} "
          f"| vùng {r['Vùng (nhị thức)']} -> {r['Kết luận']}")
worst = rate_period.drop(columns="Số phiên").drop(index="Bình thường", errors="ignore")
if len(worst):
    nrm = rate_period.loc["Bình thường"].drop("Số phiên")
    for name, row in worst.iterrows():
        v99, v95 = f"{VIOL_MODEL}_99", f"{VIOL_MODEL}_95"
        print(f"\n{name}: tỷ lệ vi phạm VaR-{VIOL_MODEL} 99% = {row[v99]:.2f}% (bình thường {nrm[v99]:.2f}%), "
              f"VaR-{VIOL_MODEL} 95% = {row[v95]:.2f}% (bình thường {nrm[v95]:.2f}%)"
              + (" -> mô hình phản ứng trễ khi cú sốc xảy ra đột ngột, vi phạm dồn vào giai đoạn khủng hoảng."
                 if row[v99] > nrm[v99] else "."))

print("\nKiểm tra ES (lỗ thực tế TB trong các ngày vi phạm so với ES dự báo TB những ngày đó):")
for _, r in bt_table.iterrows():
    if r["Vi phạm"]:
        ratio = r["Lỗ TB ngày vi phạm (%)"] / r["ES dự báo TB ngày vi phạm (%)"]       # > 1: lỗ thực tế lớn hơn ES dự báo
        verdict_es = ("ES đủ bao phủ tổn thất đuôi" if ratio <= 1 else
                      "xấp xỉ (lệch < 5%)" if ratio <= 1.05 else "ES ĐÁNH GIÁ THẤP tổn thất đuôi")
        print(f"  ES {r['Mức tin cậy']} – {r['Phương pháp']:<6}: lỗ thực tế {r['Lỗ TB ngày vi phạm (%)']:.2f}% vs ES "
              f"{r['ES dự báo TB ngày vi phạm (%)']:.2f}% (tỷ lệ {ratio:.2f}) -> {verdict_es}")
print("\nLưu ý: tham số GARCH/DCC ước lượng trên TOÀN MẪU (in-sample) nên backtest trên lạc quan hơn thực tế; "
      "kết quả ngoài mẫu (ước lượng lại theo cửa sổ trượt) nằm ở mục 4.5.")

## 4.5 Dự báo rủi ro danh mục ngoài mẫu (Out-of-sample)
**Tầng 1 – Vì sao cần?** Ở mục 4.2–4.4, tham số GARCH/DCC được ước lượng trên **toàn mẫu** rồi mới backtest ⇒ mô hình đã "nhìn thấy" chính những ngày đem ra kiểm định.
Đề bài yêu cầu so sánh ít nhất 2 mô hình **ngoài mẫu**. Quy trình:
1. Kiểm định trên `N_TEST` ngày cuối. Cứ mỗi `REFIT` ngày, **ước lượng lại** toàn bộ mô hình trên `WINDOW` ngày gần nhất.
2. Giữa hai lần ước lượng, giữ nguyên tham số và lọc tiếp $\sigma_{i,t}$, $Q_t$ ⇒ $H_t$ chỉ dùng thông tin đến $t-1$.
3. So sánh 5 phương pháp trên **cùng** các ngày kiểm định:

| Phương pháp | Biến động | Tương quan | Phân phối |
|---|---|---|---|
| **DCC-N** | GARCH biên (mô hình chọn ở TV2) | DCC(1,1) – thay đổi theo ngày | Normal |
| **DCC-t** | như trên | như trên | Student-t (ν ước lượng trong cửa sổ) |
| **CCC-N** | GARCH biên | Hằng số $\bar R$ của cửa sổ | Normal |
| **EWMA-N** | RiskMetrics λ = 0,94 cho cả ma trận hiệp phương sai | | Normal |
| **HS** | Phân vị lịch sử 250 ngày | | Không giả định |

4. **Đánh giá:** Kupiec, Christoffersen, Basel (như mục 4.3) + **hàm mất mát phân vị (quantile/tick loss)** để xếp hạng:
$QL_t = (\alpha - I_t)\,(r_{p,t} + VaR_t)$, với $I_t = 1\{r_{p,t} < -VaR_t\}$. QL trung bình càng nhỏ càng tốt; nó phạt cả vi phạm (VaR quá thấp) lẫn VaR quá cao (giữ vốn thừa).

**Tầng 2:** `fit_margin` (MLE GARCH/GJR – cùng công thức mục 2.1), `fit_dcc` (QMLE bước 2 – cùng công thức mục 3.1), `rolling_portfolio_forecast` trả về $H_t$ DCC, $H_t$ CCC, ν cho từng ngày kiểm định.
Mô hình biên mỗi mã đọc từ `TV2_GARCH/garch_selected_models.csv` (biến `SPEC`, nạp ở ô đầu notebook). Output: `oos_var_es.csv`, `oos_backtest.csv`, `TV4_out_of_sample.xlsx`, `4_oos_var_vs_loss.png`.

**Tầng 3:** Mô hình tốt: tỷ lệ vi phạm ≈ α, Kupiec & Christoffersen p > 0,05, vùng Basel Xanh, QL nhỏ nhất. DCC thắng CCC ⇒ tương quan động **có giá trị dự báo** rủi ro danh mục, không chỉ khớp trong mẫu.
Vi phạm trên hình được đánh dấu theo `VIOL_MODEL` (ô cấu hình 0.2) – cùng mô hình với hình trong mẫu ở mục 4.2.

In [ ]:
# ===================== 4.5.1 CẤU HÌNH & HÀM ƯỚC LƯỢNG (cùng công thức TV2/TV3) =====================
import time
from scipy import optimize

N_TEST = 500                                    # số ngày kiểm định
REFIT = 20                                      # ước lượng lại mỗi 20 phiên
WINDOW = min(1000, len(log_ret) - N_TEST)       # cửa sổ ước lượng (tự thu nhỏ nếu mẫu ngắn)
LAMBDA_EWMA = 0.94

print(f"Rolling: WINDOW = {WINDOW}, REFIT = {REFIT}, N_TEST = {N_TEST} "
      f"({log_ret.index[-N_TEST].date()} -> {log_ret.index[-1].date()})")
print("Mô hình biên (từ TV2):", best_name.to_dict())


def g_filter(x, r, gjr, bc):
    """σ²_t của GARCH/GJR với tham số x = [mu, omega, alpha, (gamma), beta, (nu)], σ²_0 từ backcast bc."""
    mu, om, al = x[0], x[1], x[2]
    ga = x[3] if gjr else 0.0
    be = x[4] if gjr else x[3]
    e = r - mu
    z = np.empty_like(e)
    z[0] = om + (al + 0.5 * ga) * bc + be * bc
    z[1:] = om + (al + ga * (e[:-1] < 0)) * e[:-1] ** 2
    return lfilter([1.0], [1.0, -be], z), e


def g_negll(x, r, gjr, dist):
    s2, e = g_filter(x, r, gjr, np.mean((r - x[0]) ** 2))
    s2 = np.maximum(s2, 1e-12)
    if dist == "normal":
        return 0.5 * np.sum(np.log(2 * np.pi) + np.log(s2) + e ** 2 / s2)
    nu = x[-1]
    return -np.sum(gammaln((nu + 1) / 2) - gammaln(nu / 2) - 0.5 * np.log(np.pi * (nu - 2))
                   - 0.5 * np.log(s2) - (nu + 1) / 2 * np.log1p(e ** 2 / (s2 * (nu - 2))))


def fit_margin(r, gjr, dist, x0=None):
    """MLE GARCH/GJR (giống fit_garch ở 2.1, bỏ phần sai số chuẩn cho nhanh). Trả về vector tham số."""
    v = r.var()
    bnds = [(-np.inf, np.inf), (1e-8, 10 * v), (0, 1)] + ([(-1, 1)] if gjr else []) + [(0, 1)] + ([(2.05, 200)] if dist == "t" else [])
    ib = 4 if gjr else 3
    cons = [{"type": "ineq", "fun": lambda p: 0.99999 - (p[2] + (0.5 * p[3] if gjr else 0) + p[ib])}]
    if gjr:
        cons.append({"type": "ineq", "fun": lambda p: p[2] + p[3]})
    starts = [x0] if x0 is not None else []
    starts += [[r.mean(), v * 0.05, 0.05] + ([0.05] if gjr else []) + [0.90] + ([8.0] if dist == "t" else [])]
    best = None
    for s in starts:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            o = optimize.minimize(g_negll, s, args=(r, gjr, dist), method="SLSQP", bounds=bnds, constraints=cons,
                                  options={"maxiter": 500, "ftol": 1e-9})
        if np.isfinite(o.fun) and (best is None or o.fun < best.fun):
            best = o
    return best.x


def dcc_Q_path(a, b, e, Qbar):
    """Q_t, R_t (t = 0..T-1) với Q_0 = Q̄; R_t dùng ε đến t-1."""
    T_, N_ = e.shape
    Q = np.empty((T_, N_, N_)); Q[0] = Qbar
    for t in range(1, T_):
        Q[t] = (1 - a - b) * Qbar + a * np.outer(e[t - 1], e[t - 1]) + b * Q[t - 1]
    d = 1 / np.sqrt(np.einsum("tii->ti", Q))
    return Q * d[:, :, None] * d[:, None, :]


def fit_dcc(e, x0=(0.02, 0.95)):
    Qbar_w = e.T @ e / len(e)

    def nll(th):
        a, b = th
        if a < 0 or b < 0 or a + b >= 0.9999:
            return 1e10
        R = dcc_Q_path(a, b, e, Qbar_w)
        _, logdet = np.linalg.slogdet(R)
        quad = np.einsum("ti,ti->t", e, np.linalg.solve(R, e[:, :, None])[:, :, 0])
        return 0.5 * np.sum(logdet + quad - np.einsum("ti,ti->t", e, e))

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        o = optimize.minimize(nll, x0, method="SLSQP", bounds=[(0, 0.5), (0, 0.9999)],
                              constraints=[{"type": "ineq", "fun": lambda th: 0.9999 - th[0] - th[1]}])
    return o.x, Qbar_w

In [ ]:
# ===================== 4.5.2 VÒNG LẶP ROLLING: ƯỚC LƯỢNG LẠI GARCH + DCC MỖI REFIT NGÀY =====================
def rolling_portfolio_forecast(r):
    """r: T × N lợi suất (thập phân). Trả về H_dcc, H_ccc (N_TEST × N × N), nu_t (N_TEST), bảng tham số mỗi lần refit."""
    T_, N_ = r.shape
    first = T_ - N_TEST
    H_dcc = np.empty((N_TEST, N_, N_)); H_ccc = np.empty((N_TEST, N_, N_)); nu_t = np.empty(N_TEST)
    log_rows, prev = [], {}
    for start in range(first, T_, REFIT):
        end = min(start + REFIT, T_)
        seg = r[start - WINDOW:end] * SCALE                 # cửa sổ ước lượng + các ngày cần dự báo
        win = seg[:WINDOW]
        sig = np.empty_like(seg); eps = np.empty_like(seg)
        for i, tk in enumerate(tickers):                    # Bước 1: GARCH biên từng mã
            x = fit_margin(win[:, i], **SPEC[tk], x0=prev.get(tk))
            prev[tk] = x
            s2, e = g_filter(x, seg[:, i], SPEC[tk]["gjr"], np.mean((win[:, i] - x[0]) ** 2))
            sig[:, i] = np.sqrt(s2); eps[:, i] = e / sig[:, i]
        (a, b), Qbar_w = fit_dcc(eps[:WINDOW], x0=prev.get("dcc", (0.02, 0.95)))   # Bước 2: DCC
        prev["dcc"] = (a, b)
        R = dcc_Q_path(a, b, eps, Qbar_w)                   # lọc Q_t qua cả cửa sổ + ngày dự báo
        Rbar = np.corrcoef(eps[:WINDOW].T)
        Hd = sig[:, :, None] * R * sig[:, None, :] / SCALE ** 2          # về đơn vị thập phân
        Hc = sig[:, :, None] * Rbar[None] * sig[:, None, :] / SCALE ** 2
        # ν cho DCC-t: MLE trên z_p = r_p / σ_p trong CỬA SỔ ƯỚC LƯỢNG (không dùng ngày kiểm định)
        sp_win = np.sqrt(np.einsum("i,tij,j->t", w, Hd[:WINDOW], w))
        zp = (r[start - WINDOW:start] @ w) / sp_win
        nu = optimize.minimize_scalar(std_t_nll, bounds=(2.1, 200), args=(zp,), method="bounded").x
        k = slice(start - first, end - first)
        H_dcc[k], H_ccc[k], nu_t[k] = Hd[WINDOW:], Hc[WINDOW:], nu
        log_rows.append({"Refit từ": log_ret.index[start].date(), "a": a, "b": b, "a+b": a + b, "nu_p": nu})
    return H_dcc, H_ccc, nu_t, pd.DataFrame(log_rows)


t0 = time.time()
H_dcc_oos, H_ccc_oos, nu_oos, refit_log = rolling_portfolio_forecast(log_ret.values)
print(f"✔ Xong {len(refit_log)} lần ước lượng lại trong {time.time() - t0:.0f}s")
display(refit_log.round(4))


# ===================== 4.5.3 EWMA COVARIANCE (RiskMetrics) =====================
def ewma_cov(r, lam=LAMBDA_EWMA):
    """S_t = λ S_{t-1} + (1-λ) r_{t-1} r_{t-1}'; khởi tạo bằng hiệp phương sai mẫu của cửa sổ đầu tiên."""
    T_, N_ = r.shape
    S_ = np.empty((T_, N_, N_))
    S_[0] = np.cov(r[:WINDOW].T)
    for t in range(1, T_):
        S_[t] = lam * S_[t - 1] + (1 - lam) * np.outer(r[t - 1], r[t - 1])
    return S_[-N_TEST:]


H_ewma_oos = ewma_cov(log_ret.values)

In [ ]:
# ===================== 4.5.4 VaR / ES NGOÀI MẪU CHO 5 PHƯƠNG PHÁP =====================
test_idx = log_ret.index[-N_TEST:]
sp_oos = {name: pd.Series(np.sqrt(np.einsum("i,tij,j->t", w, Hm, w)), index=test_idx)
          for name, Hm in [("DCC", H_dcc_oos), ("CCC", H_ccc_oos), ("EWMA", H_ewma_oos)]}

oos = pd.DataFrame({"r_p": var_es.loc[test_idx, "r_p"], "loss": -var_es.loc[test_idx, "r_p"],
                    "sigma_DCC": sp_oos["DCC"], "sigma_CCC": sp_oos["CCC"], "sigma_EWMA": sp_oos["EWMA"], "nu_DCC": nu_oos})
OOS_METHODS = ["DCC-N", "DCC-t", "CCC-N", "EWMA-N", "HS"]
for cl in VAR_LEVELS:
    a_lvl = 1 - cl
    tag = f"{int(cl * 100)}"
    oos[f"VaR_DCC-N_{tag}"], oos[f"ES_DCC-N_{tag}"] = var_es_normal(sp_oos["DCC"], a_lvl)
    oos[f"VaR_DCC-t_{tag}"], oos[f"ES_DCC-t_{tag}"] = var_es_t(sp_oos["DCC"], a_lvl, nu_oos)
    oos[f"VaR_CCC-N_{tag}"], oos[f"ES_CCC-N_{tag}"] = var_es_normal(sp_oos["CCC"], a_lvl)
    oos[f"VaR_EWMA-N_{tag}"], oos[f"ES_EWMA-N_{tag}"] = var_es_normal(sp_oos["EWMA"], a_lvl)
    oos[f"VaR_HS_{tag}"], oos[f"ES_HS_{tag}"] = var_es.loc[test_idx, f"VaR_HS_{tag}"], var_es.loc[test_idx, f"ES_HS_{tag}"]
assert oos.notna().all().all(), "Còn NaN trong dự báo ngoài mẫu"
oos.index.name = "Date"
oos.to_csv(OUT["TV4"] / "oos_var_es.csv", encoding="utf-8-sig")

In [ ]:
# ===================== 4.5.5 BACKTEST NGOÀI MẪU + QUANTILE LOSS =====================
def quantile_loss(r_p_, var_, a_):
    hit = (r_p_ < -var_).astype(float)
    return ((a_ - hit) * (r_p_ + var_)).mean()


oos_rows = []
for cl in VAR_LEVELS:
    tag, a_lvl = f"{int(cl * 100)}", 1 - cl
    for meth in OOS_METHODS:
        I = (oos["loss"] > oos[f"VaR_{meth}_{tag}"]).astype(int).values
        n_, x_ = len(I), int(I.sum())
        lr_uc, p_uc = kupiec_pof(I, a_lvl)
        lr_ind, p_ind, _ = christoffersen_ind(I)
        oos_rows.append({
            "Phương pháp": meth, "Mức tin cậy": f"{tag}%", "Số ngày": n_, "Vi phạm kỳ vọng": n_ * a_lvl,
            "Vi phạm thực tế": x_, "Tỷ lệ vi phạm (%)": 100 * x_ / n_,
            "Kupiec p": p_uc, "Christoffersen p": p_ind, "CC p": stats.chi2.sf(lr_uc + lr_ind, 2),
            "Basel (cả kỳ KĐ)": binomial_zone(x_, n_, a_lvl),
            "Vi phạm 250 ngày cuối": int(I[-BASEL_WINDOW:].sum()),
            "Quantile loss (x1e4)": quantile_loss(oos["r_p"], oos[f"VaR_{meth}_{tag}"], a_lvl) * 1e4,
            "VaR TB (%)": oos[f"VaR_{meth}_{tag}"].mean() * 100, "ES TB (%)": oos[f"ES_{meth}_{tag}"].mean() * 100,
            "Đánh giá": ("ĐÁNH GIÁ THẤP rủi ro" if (p_uc < ALPHA and x_ > n_ * a_lvl) else
                         "THỔI PHỒNG rủi ro" if (p_uc < ALPHA and x_ < n_ * a_lvl) else "Tần suất vi phạm phù hợp")
                        + ("; vi phạm THÀNH CỤM" if p_ind < ALPHA else "")})
oos_bt = pd.DataFrame(oos_rows)
m99 = oos_bt["Mức tin cậy"] == "99%"
oos_bt.loc[m99, "Basel (250 ngày cuối, VaR99)"] = ["Xanh" if v <= 4 else ("Vàng" if v <= 9 else "Đỏ")
                                                   for v in oos_bt.loc[m99, "Vi phạm 250 ngày cuối"]]
oos_bt["Basel (250 ngày cuối, VaR99)"] = oos_bt["Basel (250 ngày cuối, VaR99)"].fillna("")
oos_bt["Hạng QL"] = oos_bt.groupby("Mức tin cậy")["Quantile loss (x1e4)"].rank().astype(int)
display(oos_bt.round(4))
oos_bt.to_csv(OUT["TV4"] / "oos_backtest.csv", index=False, encoding="utf-8-sig")

for cl in VAR_LEVELS:
    sub = oos_bt[oos_bt["Mức tin cậy"] == f"{int(cl * 100)}%"].set_index("Phương pháp")
    print(f"VaR {int(cl * 100)}%: tốt nhất theo quantile loss = {sub['Quantile loss (x1e4)'].idxmin()} | "
          f"đạt cả Kupiec & Christoffersen: {', '.join(sub.index[(sub['Kupiec p'] >= ALPHA) & (sub['Christoffersen p'] >= ALPHA)]) or 'không có'}")


# ===================== 4.5.6 SO SÁNH TRONG MẪU vs NGOÀI MẪU (DCC) =====================
ins = bt_table.set_index(["Phương pháp", "Mức tin cậy"])
cmp_rows = []
for cl in VAR_LEVELS:
    tag = f"{int(cl * 100)}%"
    for m_in, m_out in [("Normal", "DCC-N"), ("t", "DCC-t"), ("HS", "HS")]:
        o = oos_bt[(oos_bt["Phương pháp"] == m_out) & (oos_bt["Mức tin cậy"] == tag)].iloc[0]
        cmp_rows.append({"Mức tin cậy": tag, "Phương pháp": m_out,
                         "Tỷ lệ vi phạm trong mẫu (%)": ins.loc[(m_in, tag), "Tỷ lệ vi phạm (%)"],
                         "Tỷ lệ vi phạm ngoài mẫu (%)": o["Tỷ lệ vi phạm (%)"],
                         "Kỳ vọng (%)": 100 * (1 - float(tag[:-1]) / 100)})
in_vs_out = pd.DataFrame(cmp_rows)
display(in_vs_out.round(2))
gap_in = (in_vs_out["Tỷ lệ vi phạm trong mẫu (%)"] - in_vs_out["Kỳ vọng (%)"]).abs().mean()
gap_out = (in_vs_out["Tỷ lệ vi phạm ngoài mẫu (%)"] - in_vs_out["Kỳ vọng (%)"]).abs().mean()
print(f"Sai lệch TB so với kỳ vọng: trong mẫu = {gap_in:.2f} điểm %, ngoài mẫu = {gap_out:.2f} điểm %")
print("=> Ngoài mẫu kém hơn: backtest trong mẫu đã lạc quan, nên lấy kết quả ngoài mẫu làm kết luận chính."
      if gap_out > gap_in + 0.25 else
      "=> Ngoài mẫu không kém hơn đáng kể: mô hình ổn định, kết quả trong mẫu không bị lạc quan quá mức.")

In [ ]:
import json
# ===================== 4.5.7 ĐỒ THỊ & LƯU KẾT QUẢ NGOÀI MẪU =====================
fig, axes = plt.subplots(len(VAR_LEVELS), 1, figsize=(13, 4.2 * len(VAR_LEVELS)), sharex=True)
for ax, cl in zip(np.atleast_1d(axes), VAR_LEVELS):
    tag = f"{int(cl * 100)}"
    ax.plot(test_idx, oos["loss"] * 100, color="lightgrey", lw=0.6, label="Tổn thất thực tế")
    for meth, c in zip(OOS_METHODS, ["tab:red", "tab:purple", "tab:orange", "tab:blue", "tab:green"]):
        ax.plot(test_idx, oos[f"VaR_{meth}_{tag}"] * 100, lw=0.9, color=c, label=f"VaR {meth}")
    br = oos["loss"] > oos[f"VaR_{VIOL_OOS}_{tag}"]
    ax.scatter(test_idx[br], oos["loss"][br] * 100, color="black", s=12, zorder=3, label=f"Vi phạm ({VIOL_OOS}) – {int(br.sum())}")
    shade_stress(ax, alpha=0.08, xlim=(test_idx[0], test_idx[-1]))      # chỉ tô giai đoạn nằm trong cửa sổ kiểm định
    ax.set_title(f"VaR {tag}% ngoài mẫu (ước lượng lại mỗi {REFIT} ngày, cửa sổ {WINDOW} ngày)")
    ax.set_ylabel("%"); ax.grid(alpha=0.3)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.35)                           # chừa chỗ cho chú thích
    ax.legend(fontsize=7, ncol=4, loc="upper right")
plt.tight_layout(); plt.savefig(OUT["TV4"] / "4_oos_var_vs_loss.png", dpi=150, bbox_inches="tight"); plt.show()
if not any(pd.Timestamp(e_) >= test_idx[0] for _, (s_, e_) in STRESS.items()):
    print(f"Lưu ý: cả hai giai đoạn khủng hoảng (Covid-19, Trái phiếu & lãi suất) đều nằm TRƯỚC cửa sổ ngoài mẫu "
          f"({test_idx[0].date()} → {test_idx[-1].date()}) nên không tô trên hình; kết quả ngoài mẫu phản ánh giai đoạn thị trường tương đối bình thường. "
          f"Muốn kiểm định cả khủng hoảng 2022 thì tăng N_TEST (ví dụ 700) – đổi lại cửa sổ ước lượng ngắn hơn.")

with pd.ExcelWriter(OUT["TV4"] / "TV4_out_of_sample.xlsx", engine="openpyxl") as xw:
    oos_bt.to_excel(xw, sheet_name="Backtest_ngoai_mau", index=False)
    in_vs_out.to_excel(xw, sheet_name="Trong_vs_ngoai_mau", index=False)
    refit_log.to_excel(xw, sheet_name="Tham_so_moi_lan_refit", index=False)
    oos.to_excel(xw, sheet_name="VaR_ES_ngoai_mau")
print("✔ Đã lưu kết quả ngoài mẫu TV4 vào", OUT["TV4"])

# ---- File bàn giao cho TV5 ----
with open(OUT["TV4"] / "tv4_settings.json", "w", encoding="utf-8") as fh:
    json.dump({"weights": {tk: float(x) for tk, x in zip(tickers, WEIGHTS.values)}, "N_TEST": int(N_TEST), "REFIT": int(REFIT),
               "WINDOW": int(WINDOW), "VIOL_MODEL": VIOL_MODEL}, fh, ensure_ascii=False, indent=1)
print("✔ Đã lưu file bàn giao", OUT["TV4"] / "tv4_settings.json")

### 4.5b Diễn giải kết quả ngoài mẫu (gợi ý viết báo cáo)
- So sánh **DCC-N vs CCC-N**: cùng mô hình biên, chỉ khác tương quan động/hằng ⇒ chênh lệch vi phạm & quantile loss chính là **giá trị dự báo của DCC**.
- So sánh **DCC-N vs DCC-t**: tác động của giả định đuôi dày lên VaR 99%.
- So sánh với **EWMA** và **HS**: mô hình phức tạp có thực sự tốt hơn phương pháp đơn giản ngân hàng hay dùng không?
- Bảng "trong mẫu vs ngoài mẫu": nếu tỷ lệ vi phạm ngoài mẫu cao hơn rõ rệt ⇒ backtest trong mẫu đã lạc quan; báo cáo nên lấy **kết quả ngoài mẫu làm kết luận chính**.
- Tỷ lệ vi phạm **thấp hơn** kỳ vọng với Kupiec p < 0,05 (ví dụ VaR 95% của DCC) nghĩa là mô hình **thổi phồng** rủi ro: an toàn nhưng giữ vốn thừa – quantile loss giúp xếp hạng trade-off này.
- *Giới hạn:* giai đoạn kiểm định là `N_TEST` ngày cuối (không chứa hai giai đoạn khủng hoảng của đề); chỉ 5 vi phạm kỳ vọng ở VaR 99% nên kiểm định Kupiec/Christoffersen có sức mạnh thấp.

In [ ]:
# ===================== 4.5c DIỄN GIẢI NGOÀI MẪU TỰ ĐỘNG =====================
for cl in VAR_LEVELS:
    lv = f"{int(cl * 100)}%"
    sub = oos_bt[oos_bt["Mức tin cậy"] == lv].set_index("Phương pháp")
    print(f"VaR {lv}:")
    for meth, r_ in sub.iterrows():
        print(f"  {meth:<7}: {r_['Vi phạm thực tế']} vi phạm / kỳ vọng {r_['Vi phạm kỳ vọng']:.1f} ({r_['Tỷ lệ vi phạm (%)']:.1f}%) | "
              f"Kupiec p = {r_['Kupiec p']:.3f} | QL = {r_['Quantile loss (x1e4)']:.2f} (hạng {r_['Hạng QL']}) -> {r_['Đánh giá']}")
    d = sub.loc["CCC-N", "Quantile loss (x1e4)"] - sub.loc["DCC-N", "Quantile loss (x1e4)"]
    print(f"  Giá trị dự báo của tương quan động (QL CCC-N − QL DCC-N) = {d:+.3f}"
          + (" -> DCC tốt hơn CCC." if d > 0 else " -> DCC KHÔNG tốt hơn CCC ngoài mẫu."))
    print(f"  Chênh lệch DCC-t so với DCC-N: VaR TB {sub.loc['DCC-t', 'VaR TB (%)'] - sub.loc['DCC-N', 'VaR TB (%)']:+.2f} điểm %, "
          f"QL {sub.loc['DCC-t', 'Quantile loss (x1e4)'] - sub.loc['DCC-N', 'Quantile loss (x1e4)']:+.3f}")

### TV4 – Gợi ý trả lời vấn đáp
**Câu 1 – VaR khác ES thế nào? Vì sao Basel III chuyển sang ES?** VaR chỉ cho biết **ngưỡng** tổn thất không bị vượt với xác suất 1 − α, không nói
gì về mức lỗ khi đã vượt ngưỡng; ES là **trung bình tổn thất trong đuôi** vượt VaR. VaR không có tính cộng gộp dưới (subadditivity) nên có thể
"phạt" đa dạng hoá và bỏ qua đuôi dày; ES là thước đo rủi ro nhất quán (coherent), nhạy với độ dày đuôi. Vì vậy FRTB (Basel III) dùng ES 97,5%
thay VaR 99% cho vốn rủi ro thị trường (vẫn backtest bằng VaR).

**Câu 2 – Christoffersen bị bác bỏ (p < 0,05) nghĩa là gì?** Vi phạm xảy ra **thành cụm**: sau một ngày lỗ vượt VaR, xác suất lỗ vượt VaR ngày hôm
sau cao hơn α ⇒ mô hình cập nhật biến động/tương quan quá chậm. Danh mục có thể chịu **chuỗi lỗ lớn liên tiếp** đúng khi thị trường căng thẳng,
vốn dự phòng tính theo VaR là không đủ ⇒ cần mô hình phản ứng nhanh hơn (GJR, a lớn hơn, cửa sổ ngắn hơn) hoặc cộng thêm vốn đệm.